In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

df = pd.read_excel('Zomato.xlsx')
df.shape

(9551, 18)

In [3]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 9551 entries, 0 to 9550
Data columns (total 18 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   RestaurantID          9551 non-null   int64  
 1   RestaurantName        9551 non-null   object 
 2   CountryCode           9551 non-null   int64  
 3   City                  9551 non-null   str    
 4   Address               9551 non-null   str    
 5   Locality              9551 non-null   str    
 6   LocalityVerbose       9551 non-null   str    
 7   Cuisines              9542 non-null   str    
 8   Currency              9551 non-null   str    
 9   Has_Table_booking     9551 non-null   str    
 10  Has_Online_delivery   9551 non-null   str    
 11  Is_delivering_now     9551 non-null   str    
 12  Switch_to_order_menu  9551 non-null   str    
 13  Price_range           9551 non-null   int64  
 14  Votes                 9551 non-null   int64  
 15  Average_Cost_for_two  9551 non-n

In [4]:
df.describe()

,RestaurantID,CountryCode,Price_range,Votes,Average_Cost_for_two,Rating
count,9.551000e+03,9551.000000,9551.000000,9551.000000,9551.000000,9551.000000
mean,9.051128e+06,18.365616,1.804837,156.909748,1199.210763,2.891268
std,8.791521e+06,56.750546,0.905609,430.169145,16121.183073,1.128845
min,5.300000e+01,1.000000,1.000000,0.000000,0.000000,1.000000
25%,3.019625e+05,1.000000,1.000000,5.000000,250.000000,2.500000
50%,6.004089e+06,1.000000,2.000000,31.000000,400.000000,3.200000
75%,1.835229e+07,1.000000,2.000000,131.000000,700.000000,3.700000
max,1.850065e+07,216.000000,4.000000,10934.000000,800000.000000,4.900000


In [5]:
df.duplicated().sum()

np.int64(0)

In [6]:
df.isnull().sum()

RestaurantID            0
RestaurantName          0
CountryCode             0
City                    0
Address                 0
Locality                0
LocalityVerbose         0
Cuisines                9
Currency                0
Has_Table_booking       0
Has_Online_delivery     0
Is_delivering_now       0
Switch_to_order_menu    0
Price_range             0
Votes                   0
Average_Cost_for_two    0
Rating                  0
CountryName             0
dtype: int64

In [7]:
df.isna().sum()

RestaurantID            0
RestaurantName          0
CountryCode             0
City                    0
Address                 0
Locality                0
LocalityVerbose         0
Cuisines                9
Currency                0
Has_Table_booking       0
Has_Online_delivery     0
Is_delivering_now       0
Switch_to_order_menu    0
Price_range             0
Votes                   0
Average_Cost_for_two    0
Rating                  0
CountryName             0
dtype: int64

In [8]:
for col in ['CountryName', 'Currency']:
    print(f"Unique values in {col}: {df[col].unique()}")

Unique values in CountryName: <StringArray>
[         'Philippines',               'Brazil',        'United States',
            'Australia',               'Canada',            'Singapore',
 'United Arab Emirates',                'India',            'Indonesia',
          'New Zealand',              'England',                'Qatar',
         'South Africa',            'Sri Lanka',               'Turkey']
Length: 15, dtype: str
Unique values in Currency: <StringArray>
[      'Botswana Pula(P)',     'Brazilian Real(R$)',              'Dollar($)',
     'Emirati Diram(AED)',     'Indian Rupees(Rs.)', 'Indonesian Rupiah(IDR)',
          'NewZealand($)',              'Pounds(?)',        'Qatari Rial(QR)',
                'Rand(R)',  'Sri Lankan Rupee(LKR)',       'Turkish Lira(TL)']
Length: 12, dtype: str


**Insights:** There are `15 Countries` and only `12 currencies`. Need to check those values. Which are all the countries share the same currency as per the dataset

In [9]:
df.groupby('Currency')['CountryName'].unique().apply(sorted)

Currency
Botswana Pula(P)                                          [Philippines]
Brazilian Real(R$)                                             [Brazil]
Dollar($)                 [Australia, Canada, Singapore, United States]
Emirati Diram(AED)                               [United Arab Emirates]
Indian Rupees(Rs.)                                              [India]
Indonesian Rupiah(IDR)                                      [Indonesia]
NewZealand($)                                             [New Zealand]
Pounds(?)                                                     [England]
Qatari Rial(QR)                                                 [Qatar]
Rand(R)                                                  [South Africa]
Sri Lankan Rupee(LKR)                                       [Sri Lanka]
Turkish Lira(TL)                                               [Turkey]
Name: CountryName, dtype: object

**Insight:** `4 Countries` Australia, Canada, Singapore and United States `share same currency` **(Dollar $)** but if we are doing `currency normalization.` we need to find the individual currency dollar conversion.
Also `Philippines is Peso`, but here it is mentioned as `Pula`. Which also need to be considered. Mostly this should be the data error

In [10]:
currency_count = df[['CountryName','Currency']].value_counts().sort_values(ascending=False)

currency_pct = (df[['CountryName','Currency']].value_counts(normalize=True) *100).round(2)

currency_df = pd.DataFrame({
    "Count": currency_count,
    "Percentage": currency_pct
})

print(currency_df)  

                                             Count  Percentage
CountryName          Currency                                 
India                Indian Rupees(Rs.)       8652       90.59
United States        Dollar($)                 434        4.54
England              Pounds(?)                  80        0.84
Brazil               Brazilian Real(R$)         60        0.63
United Arab Emirates Emirati Diram(AED)         60        0.63
South Africa         Rand(R)                    60        0.63
New Zealand          NewZealand($)              40        0.42
Turkey               Turkish Lira(TL)           34        0.36
Australia            Dollar($)                  24        0.25
Philippines          Botswana Pula(P)           22        0.23
Indonesia            Indonesian Rupiah(IDR)     21        0.22
Singapore            Dollar($)                  20        0.21
Qatar                Qatari Rial(QR)            20        0.21
Sri Lanka            Sri Lankan Rupee(LKR)      20     

**Insight**: India dominates the chart with 90.59% of data with n=8652. All our analysis will be influenced by Indian market. So we need to consider only Indian market when we are answering questions which involves currency, price and avg_cost_of_two. 

In [11]:
df['online_delivery_flag'] = df['Has_Online_delivery'].map({'Yes': 1, 'No': 0})

pd.crosstab(df['online_delivery_flag'], df['Has_Online_delivery'], margins=True, margins_name='Total')

Has_Online_delivery,No,Yes,Total
online_delivery_flag,,,
0,7100,0,7100
1,0,2451,2451
Total,7100,2451,9551


In [12]:
print(f'Overall Online Adoption Rate: {df["online_delivery_flag"].mean().round(2)}')


Overall Online Adoption Rate: 0.26


In [13]:
delivery = df.groupby('CountryName')['online_delivery_flag'].aggregate(['mean', 'count'])

delivery['mean'] = (delivery['mean']*100).round(2)

delivery.columns = ['Online Delivery Adoption Rate (%)', 'Restaurants Count']

delivery.sort_values(by='Online Delivery Adoption Rate (%)', ascending=False, inplace=True)

print(delivery)

                      Online Delivery Adoption Rate (%)  Restaurants Count
CountryName                                                               
United Arab Emirates                              46.67                 60
India                                             28.01               8652
Brazil                                             0.00                 60
Canada                                             0.00                  4
England                                            0.00                 80
Indonesia                                          0.00                 21
Australia                                          0.00                 24
New Zealand                                        0.00                 40
Philippines                                        0.00                 22
Singapore                                          0.00                 20
Qatar                                              0.00                 20
South Africa             

**Insight:** Eventhough UAE tops the chart with highest online adoption, India has huge potential as it has 8652 restaurants listed, Zomato need to incentivise the restaurant owners to shift to online delivery. Other markets like US also has 434 restaurants, second highest in the list as the second focus market to expand and convert those to online delivery.

In [14]:
df.groupby('Votes')['Rating'].value_counts().head(10)

Votes  Rating
0      1.0       1094
1      1.0        483
2      1.0        327
3      1.0        244
4      3.0         89
       2.9         81
       3.1         14
       2.8         10
       2.7          4
       2.6          3
Name: count, dtype: int64

In [15]:
df.groupby('Rating')['Votes'].value_counts()

Rating  Votes
1.0     0        1094
        1         483
        2         327
        3         244
1.8     322         1
                 ... 
4.9     40          1
        408         1
        194         1
        522         1
        1042        1
Name: count, Length: 3713, dtype: int64

In [16]:
df[df['Votes']<4]['Rating'].value_counts()

Rating
1.0    2148
Name: count, dtype: int64

**Insight:** When the votes are 0, it has created a placeholder value of rating as 1.0. also for all restaurants with votes <4 considered the placeholder rating of 1.0. 

In [18]:
# create and export zomato_global.csv with one row per restaurant and these columns:

# RestaurantID, CountryName, City, Currency, online_delivery_flag, is_rated

df[['RestaurantID', 'CountryName', 'City', 'Currency', 'online_delivery_flag', 'Rating']].rename(columns={'Rating': 'is_rated'}).to_csv('zomato_global.csv', index=False)

In [ ]:
df['is_rated'] = (df['Votes'] >= 4).astype(int)


In [20]:
df['rating_clean'] = df['Rating'].where(df['Votes'] >= 4)
df['cost_clean'] = df['Average_Cost_for_two'].where(df['Average_Cost_for_two'] > 0)
df['table_booking_flag'] = df['Has_Table_booking'].map({'Yes': 1, 'No': 0})
df['cuisines_clean'] = df['Cuisines'].fillna('Not Specified')
df['cuisine_count'] = df['cuisines_clean'].str.split('|').str.len()
df['cuisine_group'] = df['cuisine_count'].clip(upper=5)

In [21]:
df[['RestaurantID', 'CountryName', 'City', 'Currency', 'online_delivery_flag', 'is_rated']].to_csv('zomato_global.csv', index=False)

In [22]:
df_india = df[df['CountryName'] == 'India'].drop(columns=[
    'CountryCode', 'CountryName', 'Currency', 'Address', 'LocalityVerbose', 'Cuisines',
    'Has_Table_booking', 'Has_Online_delivery', 'Is_delivering_now', 'Switch_to_order_menu'])
df_india.shape

(8652, 16)

In [23]:
cuisine_india = df_india[['RestaurantID', 'cuisines_clean']].copy()
cuisine_india['Cuisine'] = cuisine_india['cuisines_clean'].str.split('|')
cuisine_india = cuisine_india.explode('Cuisine')
cuisine_india['Cuisine'] = cuisine_india['Cuisine'].str.strip()
cuisine_india = cuisine_india[['RestaurantID', 'Cuisine']]
cuisine_india.shape

(17854, 2)

In [24]:
df_india.to_csv('zomato_india.csv', index=False)
cuisine_india.to_csv('zomato_india_cuisines.csv', index=False)